In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('data/CALCOFI_DIC_20250122.csv') 
df.head()

,EXPOCODE,Ship_Name,Year_UTC,Month_UTC,Day_UTC,Time_UTC,Station_ID,Latitude,Longitude,Depth,CTDTEMP_ITS90,CTDTEMP_flag,Salinity_PSS78,Salinity_flag,DIC,DIC_flag,TA,TA_flag
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,decimal degrees,decimal degres,m,deg_C,NaN,NaN,NaN,umol/kg,NaN,umol/kg,NaN
1,31EB19830318,ELLEN B. SCRIPPS,1983.0,3.0,19.0,0:00:00,090.0 062.0,32.3513,-120.0954,10,15.36,2.0,-999.0,9.0,1989.4,2.0,-999,9.0
2,31EB19830318,ELLEN B. SCRIPPS,1983.0,3.0,19.0,0:00:00,090.0 062.0,32.3513,-120.0954,10,-999,9.0,-999.0,9.0,1989.8,2.0,-999,9.0
3,31EB19830318,ELLEN B. SCRIPPS,1983.0,3.0,19.0,0:00:00,090.0 062.0,32.3513,-120.0954,10,-999,9.0,-999.0,9.0,1989.3,2.0,-999,9.0
4,31EB19830318,ELLEN B. SCRIPPS,1983.0,3.0,19.0,0:00:00,090.0 062.0,32.3513,-120.0954,10,-999,9.0,-999.0,9.0,1988.6,2.0,-999,9.0



## Step 1: Understand the Big Picture
NoteQuestion 1
Before touching the data, answer the following about the Global Air Pollution dataset:

- What is the response variable? Is this a regression or classification problem?
AQI Value - regression problem

- What performance metric will you use, and why is it appropriate?

 RMSE (Root Mean Squared Error), since it penalizes large errors more strongly, which matters for AQI because extreme pollution values are the most important to predict accurately.

It does is measured in the same units as AQI, making interpretation intuitive, and is standard for environmental and continuous‑value prediction tasks.

- What is a sensible baseline to beat (e.g., predicting the mean, a simple rule, a published benchmark)?

Predicting the mean AQI Value across all observations.

- Are there any domain-specific constraints on errors? Is over-prediction or under-prediction more costly?
in air‑quality applications, under‑prediction is more harmful.

In [ ]:
# Explore the dataset
df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23463 entries, 0 to 23462
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Country             23036 non-null  object
 1   City                23462 non-null  object
 2   AQI Value           23463 non-null  int64 
 3   AQI Category        23463 non-null  object
 4   CO AQI Value        23463 non-null  int64 
 5   CO AQI Category     23463 non-null  object
 6   Ozone AQI Value     23463 non-null  int64 
 7   Ozone AQI Category  23463 non-null  object
 8   NO2 AQI Value       23463 non-null  int64 
 9   NO2 AQI Category    23463 non-null  object
 10  PM2.5 AQI Value     23463 non-null  int64 
 11  PM2.5 AQI Category  23463 non-null  object
dtypes: int64(5), object(7)
memory usage: 2.1+ MB


,AQI Value,CO AQI Value,Ozone AQI Value,NO2 AQI Value,PM2.5 AQI Value
count,23463.000000,23463.000000,23463.000000,23463.000000,23463.000000
mean,72.010868,1.368367,35.193709,3.063334,68.519755
std,56.055220,1.832064,28.098723,5.254108,54.796443
min,6.000000,0.000000,0.000000,0.000000,0.000000
25%,39.000000,1.000000,21.000000,0.000000,35.000000
50%,55.000000,1.000000,31.000000,1.000000,54.000000
75%,79.000000,1.000000,40.000000,4.000000,79.000000
max,500.000000,133.000000,235.000000,91.000000,500.000000


In [10]:
# Find the percentage of missing values in each column
(df.isna().mean() * 100)

Country               1.819887
City                  0.004262
AQI Value             0.000000
AQI Category          0.000000
CO AQI Value          0.000000
CO AQI Category       0.000000
Ozone AQI Value       0.000000
Ozone AQI Category    0.000000
NO2 AQI Value         0.000000
NO2 AQI Category      0.000000
PM2.5 AQI Value       0.000000
PM2.5 AQI Category    0.000000
dtype: float64

- How many numerical features and how many categorical features are there?

5 numerical and 7 categorical

- Which columns (if any) have missing values? What percentage of observations are missing in each
City and Country

- Do the ranges in df.describe() look physically reasonable? Flag anything surprising.

CO AQI Value max = 133  
This is unusually high because CO AQI rarely exceeds ~50 in most cities.
It may reflect: 
a real extreme event
a reporting anomaly
a unit conversion issue
a data entry error


## Building a Numeric Pipeline
The simplest pipeline chains steps in a list of (name, object) tuples:

## Step 3: Create a representative test set and lock it away

In [17]:
from sklearn.model_selection import train_test_split

# Define the response variable and features

X = df.drop(columns=['AQI Value'])
y = df['AQI Value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training   : {X_train.shape[0]} rows')
print(f'Test       : {X_test.shape[0]} rows')

Training   : 18770 rows
Test       : 4693 rows


## Step 4: Explore the train data to gain insights
Build your own exploratory analysis on the training set. Explore the following for each variable:

Name
Type (categorical, int/float, bounded/unbounded, text, structured, etc.)
% of missing values
Noisiness and type of noise (outliers, rounding errors, etc.)
Possibly useful for the task?
Type of distribution

### Question 3
Which predictors appear most strongly related to the target? You can use corr() to find out. Describe the direction.
Are there too many predictors